# HW5: UFO Sightings in the United States

This notebook builds the two Altair (vega-lite) charts used on my Jekyll page for Homework 5.

* **Data:** NUFORC UFO sightings (scrubbed, geocoded), read directly from the course data repository URL
* **Plot 1:** sightings per year, split by the reported shape (interactive legend + tooltips)
* **Plot 2:** sightings by month and hour of day (dropdown filter by shape + tooltips)

Both charts are saved as JSON into `assets/json/` so they can be embedded with a `<vegachart>` tag.

In [1]:
import pandas as pd
import altair as alt

alt.__version__

'6.3.0'

## 1. Read in the data from a URL

In [2]:
url = 'https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/ufo-scrubbed-geocoded-time-standardized-00.csv'

# the file has no header row, so we name the columns ourselves
cols = ['date', 'city', 'state', 'country', 'shape', 'duration_seconds',
        'duration', 'comment', 'report_date', 'latitude', 'longitude']

ufos = pd.read_csv(url, header=None, names=cols, low_memory=False)
print(len(ufos), 'rows')
ufos.head()

80332 rows


,date,city,state,country,shape,duration_seconds,duration,comment,report_date,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700.0,45 minutes,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611


## 2. Data transformations

A few cleaning steps before plotting:

1. **Dates:** a handful of times are written as `24:00`, which pandas cannot parse, so I rewrite them as `00:00` and then convert the column with `pd.to_datetime`. From the timestamp I pull out the year, the month, and the hour of the day.
2. **Location:** I keep only sightings with `country == 'us'` (about 81% of the rows) so that the time of day means the same thing across the dataset.
3. **Years:** I keep 1995 through 2013. Before 1995 there are only a few hundred reports per year (NUFORC launched its online reporting form in the mid 1990s), and the 2014 data stops in early May, so it would look like a fake drop.
4. **Shapes:** there are almost 30 shape labels. I keep the six most common *descriptive* shapes and group everything else (including `other`, `unknown`, and missing values) into `other`, so the color legend stays readable.
5. **Aggregation:** the raw table has about 80,000 rows, which would make the JSON files very large. I aggregate in pandas first (counts per year and shape, and counts per month, hour, and shape) so each saved chart only contains a few hundred to a couple thousand rows.

In [3]:
ufos['date'] = pd.to_datetime(ufos['date'].str.replace(' 24:', ' 00:', regex=False),
                              format='%m/%d/%Y %H:%M', errors='coerce')
ufos['year'] = ufos['date'].dt.year
ufos['month'] = ufos['date'].dt.month
ufos['hour'] = ufos['date'].dt.hour

us = ufos[(ufos['country'] == 'us') & (ufos['year'].between(1995, 2013))].copy()
print(len(us), 'US sightings between 1995 and 2013')

top_shapes = (us.loc[~us['shape'].isin(['other', 'unknown']), 'shape']
                .value_counts().head(6).index.tolist())
us['shape_group'] = us['shape'].where(us['shape'].isin(top_shapes), 'other')
print(top_shapes)
us['shape_group'].value_counts()

57076 US sightings between 1995 and 2013
['light', 'triangle', 'circle', 'fireball', 'sphere', 'disk']


shape_group
other       22244
light       12329
triangle     5730
circle       5360
fireball     4802
sphere       3771
disk         2840
Name: count, dtype: int64

## 3. Plot 1: sightings per year by shape

**Encodings:** year on the x axis (ordinal, since these are whole years), number of sightings on the y axis (quantitative), and shape as color (nominal).

**Colors:** shape is a categorical variable with no natural order, so I use a categorical scheme (`tableau10`) where each shape gets a clearly different hue. The `other` group is drawn in gray so it reads as background.

**Interactivity:** clicking a shape in the legend highlights that line and fades the others (shift + click selects more than one). Hovering shows a tooltip with the exact year, shape, and count.

In [4]:
per_year = (us.groupby(['year', 'shape_group']).size()
              .reset_index(name='sightings'))

shape_order = top_shapes + ['other']
shape_colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f', '#edc948', '#bab0ac']

legend_select = alt.selection_point(fields=['shape_group'], bind='legend')

plot1 = alt.Chart(per_year).mark_line(point=True, strokeWidth=2.5).encode(
    x=alt.X('year:O', title='Year', axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('sightings:Q', title='Number of sightings'),
    color=alt.Color('shape_group:N', title='Shape (click to highlight)',
                    legend=alt.Legend(titleLimit=250),
                    scale=alt.Scale(domain=shape_order, range=shape_colors),
                    sort=shape_order),
    opacity=alt.condition(legend_select, alt.value(1.0), alt.value(0.12)),
    tooltip=[alt.Tooltip('year:O', title='Year'),
             alt.Tooltip('shape_group:N', title='Shape'),
             alt.Tooltip('sightings:Q', title='Sightings')]
).add_params(legend_select).properties(
    width='container', height=380,
    title='US UFO sightings per year by reported shape (1995 to 2013)'
)

plot1

alt.Chart(...)

In [5]:
plot1.save('../assets/json/ufo_sightings_by_year.json')

## 4. Plot 2: when do people see UFOs?

**Encodings:** hour of day on the x axis (ordinal), month on the y axis (ordinal), and the number of sightings as the color of each cell (quantitative). A heatmap makes two time patterns visible at once.

**Colors:** the count is a quantitative variable that goes from low to high, so I use a sequential colormap (`viridis`). It is perceptually uniform and still readable for people with common types of color blindness. Brighter cells mean more sightings.

**Interactivity:** a dropdown menu filters the heatmap to a single shape, so you can check whether, for example, fireballs follow a different pattern than lights. Hovering shows the month, hour, and count.

In [6]:
per_hour = (us.groupby(['month', 'hour', 'shape_group']).size()
              .reset_index(name='sightings'))
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
per_hour['month_name'] = per_hour['month'].map(lambda m: month_names[m - 1])

shape_dropdown = alt.binding_select(options=['all'] + shape_order,
                                    name='Reported shape: ')
shape_param = alt.param(value='all', bind=shape_dropdown)

plot2 = alt.Chart(per_hour).transform_filter(
    (shape_param == 'all') | (alt.datum.shape_group == shape_param)
).mark_rect().encode(
    x=alt.X('hour:O', title='Hour of day (0 = midnight)', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('month_name:O', title='Month', sort=month_names),
    color=alt.Color('sum(sightings):Q', title='Sightings',
                    scale=alt.Scale(scheme='viridis')),
    tooltip=[alt.Tooltip('month_name:O', title='Month'),
             alt.Tooltip('hour:O', title='Hour'),
             alt.Tooltip('sum(sightings):Q', title='Sightings')]
).add_params(shape_param).properties(
    width='container', height=380,
    title='US UFO sightings by month and hour of day (1995 to 2013)'
)

plot2

alt.Chart(...)

In [7]:
plot2.save('../assets/json/ufo_sightings_heatmap.json')

## 5. A few numbers used in the write up

In [8]:
print('Peak year:', per_year.groupby('year')['sightings'].sum().idxmax())
by_hour = us.groupby('hour').size()
print('Share of sightings between 8pm and midnight: %.1f%%' % (100 * by_hour.loc[20:23].sum() / by_hour.sum()))
by_day = us.groupby([us['date'].dt.month, us['date'].dt.day]).size().sort_values(ascending=False)
print('Busiest calendar days (month, day):')
print(by_day.head(3))

Peak year: 2012
Share of sightings between 8pm and midnight: 49.3%
Busiest calendar days (month, day):
date  date
7     4       1040
1     1        373
8     15       373
dtype: int64
